# Stage 1d — Is `stats_rewatched` built from `my_rewatching_ep`?

**Question.** Three columns carry information about rewatching, and it is not clear how they relate:

| Column | Table | Level | What we think it is |
|---|---|---|---|
| `my_rewatching` | list | one row = one user and one anime | A flag: 1 if the user is rewatching that anime now |
| `my_rewatching_ep` | list | one row = one user and one anime | A number of episodes. The author writes about the "number of rewatched episodes" that many users filled in wrongly, but does not name the column, so this is the likely candidate |
| `stats_rewatched` | user | one row = one user | A total for the user, meaning still unknown |

**The idea to test.** If `stats_rewatched` is the total of `my_rewatching_ep` over a user's anime, then (1) users with `stats_rewatched > 0` should have rows with `my_rewatching_ep > 0`, and (2) the sum should be close to `stats_rewatched`.

**What the numbers from stage 1b and 1c already say.** The 99.9th percentile of `my_rewatching_ep` is 0, so **at most about 80,000 of the 80 million rows** can have a value above 0. But about **124,000 users** have `stats_rewatched > 0`. A user needs at least one row above 0 to get a sum above 0, so there are not enough such rows for every one of those users. If this reasoning is right, `stats_rewatched` cannot simply be the sum of `my_rewatching_ep`. The cells below test it directly instead of trusting the argument.

**How to run on Kaggle.** Add the *MyAnimeList Dataset* as input and run all cells. It reads three columns of the 80M-row list table and takes a few minutes.

In [1]:
from pathlib import Path
import sys

import pandas as pd

# Works both on Kaggle and inside the repo (with the data placed in data/raw/)
KAGGLE_INPUT = Path("/kaggle/input")
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "environment.yml").exists()), None)
if KAGGLE_INPUT.exists():
    DATA_DIR = KAGGLE_INPUT
elif ROOT is not None:
    DATA_DIR = ROOT / "data" / "raw"
else:
    raise RuntimeError("Data not found: run on Kaggle or inside the repository.")

paths = {p.name: p for p in DATA_DIR.rglob("*.csv")}
na_kwargs = dict(keep_default_na=False, na_values=[""])  # only an empty cell is missing

pd.set_option("display.width", 200)
print("Python:", sys.version.split()[0], "| pandas:", pd.__version__)

users = pd.read_csv(paths["UserList.csv"], usecols=["username", "stats_rewatched"], **na_kwargs)
anime = pd.read_csv(paths["AnimeList.csv"], usecols=["anime_id", "episodes"], **na_kwargs)

cols = {"username": "category", "anime_id": "int32", "my_rewatching": "float32", "my_rewatching_ep": "int32"}
rows = pd.read_csv(paths["UserAnimeList.csv"], usecols=list(cols), dtype=cols, **na_kwargs)
print(f"list rows: {len(rows):,} | users: {len(users):,}")

Python: 3.13.18 | pandas: 2.3.3
list rows: 80,076,112 | users: 302,675


## 1. Which rows have a rewatch episode value?

In [2]:
positive = rows[rows["my_rewatching_ep"] > 0].copy()
print(f"rows with my_rewatching_ep > 0: {len(positive):,} ({len(positive) / len(rows):.3%} of all rows)")
print(f"users with at least one such row: {positive['username'].nunique():,}")
print("\nvalues among those rows:")
print(positive["my_rewatching_ep"].describe(percentiles=[.25, .5, .75, .9, .99]).round(1).to_string())

print("\nmy_rewatching on those rows:")
print(positive["my_rewatching"].value_counts(dropna=False).to_string())
print("\nrows with my_rewatching = 1 that have my_rewatching_ep = 0:",
      int(((rows["my_rewatching"] == 1) & (rows["my_rewatching_ep"] == 0)).sum()))

# Is the value ever larger than the number of episodes of that anime? (only where the episode count is known)
checked = positive.merge(anime, on="anime_id", how="left")
known = checked[checked["episodes"] > 0]
print(f"\nrows compared with the anime's episode count: {len(known):,}")
print("my_rewatching_ep > episodes of the anime:", int((known["my_rewatching_ep"] > known["episodes"]).sum()),
      f"({(known['my_rewatching_ep'] > known['episodes']).mean():.1%} of the compared rows)")

rows with my_rewatching_ep > 0: 11,964 (0.015% of all rows)
users with at least one such row: 4,614

values among those rows:
count       11964.0
mean        49572.5
std        907586.6
min             1.0
25%             2.0
50%            10.0
75%            22.0
90%            32.0
99%           194.5
max      16777215.0

my_rewatching on those rows:
my_rewatching
0.0    10488
1.0     1054
NaN      422

rows with my_rewatching = 1 that have my_rewatching_ep = 0: 44796

rows compared with the anime's episode count: 11,946
my_rewatching_ep > episodes of the anime: 259 (2.2% of the compared rows)


### Observations for part 1

**TODO (fill in after the run):** How many rows have a value above 0, and how many users? Are the values plausible episode counts, or are they dominated by huge numbers? What share of them is larger than the anime's number of episodes? How do they relate to the flag `my_rewatching`?

## 2. Do the two columns agree for each user?

In [3]:
per_user = pd.DataFrame({
    "rows_with_ep": positive.groupby("username", observed=True).size(),
    # huge values would dominate a sum, so each row is capped at 1,000 episodes
    "sum_ep_capped": positive["my_rewatching_ep"].clip(upper=1000).groupby(positive["username"], observed=True).sum(),
})
per_user.index = per_user.index.astype(object)
table = users.set_index("username").join(per_user, how="left").fillna({"rows_with_ep": 0, "sum_ep_capped": 0})
table = table.dropna(subset=["stats_rewatched"])
table["stats_positive"] = table["stats_rewatched"] > 0
table["ep_positive"] = table["rows_with_ep"] > 0

print(f"users compared: {len(table):,}")
print("\nusers with stats_rewatched > 0 (rows) against users with a my_rewatching_ep > 0 row (columns):")
print(pd.crosstab(table["stats_positive"], table["ep_positive"], margins=True).to_string())

both = table[table["stats_positive"] & table["ep_positive"]]
ratio = both["sum_ep_capped"] / both["stats_rewatched"]
print(f"\nusers positive in both: {len(both):,}")
print("rank correlation of stats_rewatched with the capped sum:",
      round(both["stats_rewatched"].rank().corr(both["sum_ep_capped"].rank()), 3))
print(f"median ratio (capped sum / stats_rewatched): {ratio.median():.3f} | "
      f"middle 90%: {ratio.quantile(.05):.3f} to {ratio.quantile(.95):.3f}")
print("equal:", f"{(both['sum_ep_capped'] == both['stats_rewatched']).mean():.1%}",
      "| stats_rewatched lower than the sum:", f"{(both['stats_rewatched'] < both['sum_ep_capped']).mean():.1%}",
      "| stats_rewatched higher than the sum:", f"{(both['stats_rewatched'] > both['sum_ep_capped']).mean():.1%}")

users compared: 302,546

users with stats_rewatched > 0 (rows) against users with a my_rewatching_ep > 0 row (columns):
ep_positive      False  True     All
stats_positive                      
False           177510   740  178250
True            120425  3871  124296
All             297935  4611  302546

users positive in both: 3,871
rank correlation of stats_rewatched with the capped sum: 0.11
median ratio (capped sum / stats_rewatched): 1.000 | middle 90%: 0.034 to 23.833
equal: 2.2% | stats_rewatched lower than the sum: 47.8% | stats_rewatched higher than the sum: 50.0%


### Observations for part 2

**TODO (fill in after the run):** In the table, how many users have `stats_rewatched > 0` but no row with a rewatch episode value? Where the two are both above 0, is `stats_rewatched` lower than, equal to, or higher than the sum? Write what the numbers show; do not decide the meaning before reading them.

## Reading the result

| What we see | What it would mean |
|---|---|
| Almost all users with `stats_rewatched > 0` have no row with `my_rewatching_ep > 0` | `stats_rewatched` is not the total of `my_rewatching_ep`. It counts something that the list table does not hold. |
| Both above 0 for most users and the sum is close to `stats_rewatched` | `stats_rewatched` is the total of rewatched episodes, and `my_rewatching_ep` is the per-anime count. |
| Sum much larger than `stats_rewatched` for most users | The two columns count different things, or `my_rewatching_ep` is full of wrong values (the author warns about this). |